# KAT-VAD v2 — **Nexar zero-shot: every T2-trained arm on `nexar_val`** (A0, A1, A2, A3, A3-ign)

No training. The 25 T2-trained checkpoints (E3's A0–A3 × 5 seeds, `dvs_anchor_mode=span`; E4-0's A3-ign × 5 seeds,
`ignore`) are scored on `nexar_val` (225 videos, 112 positive) with **`core.tools.nexar_eval`** — the exact protocol
of the A3-ign zero-shot already read for §28 D-N13 (N-B: `s1[::3]` per item, CRN per item with the arm's **T2**
statistics, native frames, seed-averaged).

**Descriptive only.** `nexar_val` is not sealed and nothing here was pre-registered as a decision: these numbers decide
nothing and `B` stays A3-ign. `nexar_test` is not read.

| step | what | GPU? |
|---|---|---|
| 0 | setup | — |
| 1 | preflight: 25 checkpoints (J10 + arm config), the three T2 bakes (stats), Nexar caches for `nexar_val` | no |
| 2 | stage `nexar_val` CLIP s1 + V2-S s1 to VM disk | no |
| 3 | score each arm (5 seeds, seed-averaged) → `REPORTS/nexar_val_zeroshot/<arm>/` | yes |
| 4 | regression gate: fresh A3-ign == the saved §28 zero-shot read-out | no |
| 5 | summary table + paired-by-video contrasts (printed) → paste `zeroshot_summary.md` to Claude | no |

Every step skips what is already on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: v1 data/cache roots (env only)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/, data/, outputs/v2_e3, v2_e4_0; write: outputs/v2/REPORTS
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/nexar_eval.py').is_file(), 'this upload predates Nexar N2-N5 -- re-upload branch v2'

T2       = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig' (the training corpus)
SEEDS    = list(constants.V2_E3_SEEDS)                 # 2024-2028 (E3 and E4-0 trained the same five)
STEP     = constants.NEXAR_STEP_BUDGET                 # 1,740 = E3 / E4-0's last step (J10)
CRN      = 'R2'
V2S      = constants.VIDEOMAE_ENCODER_S                # 'vit_s_k710_dl_from_giant'
SPAN, IGN = constants.DVS_ANCHOR_MODE_SPAN, constants.DVS_ANCHOR_MODE_IGNORE
OFF      = constants.V2_OFF
ARMS     = ['A0', 'A1', 'A2', 'A3', 'A3ign']
EXPECT   = {'A0': (OFF, OFF, SPAN), 'A1': (CRN, OFF, SPAN), 'A2': (OFF, V2S, SPAN),
            'A3': (CRN, V2S, SPAN), 'A3ign': (CRN, V2S, IGN)}            # (v2.crn, v2.motion, dvs_anchor_mode)

V2_CACHE = Path(DRIVE_V2) / 'cache'
BAKE     = {'A1': f'A1_{CRN}', 'A2': 'A2_S', 'A3': f'A3_{CRN}_S', 'A3ign': f'A3_{CRN}_S'}
STATS    = {arm: V2_CACHE / 'v2' / tag / T2 for arm, tag in BAKE.items()}   # each arm's own T2 statistics
RUNS     = {**{a: constants.OUTPUT_ROOT / 'v2_e3' / a for a in ('A0', 'A1', 'A2', 'A3')},
            'A3ign': constants.OUTPUT_ROOT / 'v2_e4_0' / constants.V2_E4_0_ARM}
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / V2S / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
DATA     = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WHOLE}'   # defs.json
REPORTS  = constants.OUTPUT_ROOT / 'v2' / 'REPORTS'
CENSUS   = REPORTS / 'nexar_n0' / 'census.json'
OUT      = REPORTS / 'nexar_val_zeroshot'
SAVED_ZS = REPORTS / f'nexar_val_{IGN}' / f'zeroshot_e4_0_{constants.V2_E4_0_ARM}'   # §28 D-N13's read
LOCAL    = Path('/content/nexar_zs')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('CLIP_S1', 'VIDEO_S1', 'DATA', 'CENSUS', 'OUT', 'SAVED_ZS'):
    print(f'{name:9s}', globals()[name])
for arm in ARMS:
    print(f'{arm:6s} runs {RUNS[arm]}  stats {STATS.get(arm, "(plain CLIP)")}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 3 will be slow'

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) to VM disk once (bit-identical; C2).'''
    dst.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for v in ids or []:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for pattern in ('*.json', '*.npz'):
        for m in src.glob(pattern):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} ({len(ids or [])} ids) in {time.time() - t0:.0f}s')
    return dst


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight

J10 on every checkpoint (its `global_step` == the last line of its `metrics.jsonl` == 1,740) and each arm's config
(`v2.crn`, `v2.motion`, `loss.dvs_anchor_mode`). Each bake's manifest must name its arm, and all three bakes must share
one T2 train split. `nexar_test` is never loaded.

In [ ]:
from core.data.v2_splits import load_split

for arm in ARMS:
    for s in SEEDS:
        d = RUNS[arm] / f's{s}'
        step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
        assert last is not None and step == last['global_step'] == STEP, f'{arm} s{s}: J10 (ckpt {step})'
        cfg = yaml.safe_load((d / 'config.yaml').read_text())
        got = (cfg['v2']['crn'], cfg['v2']['motion'], cfg['loss']['dvs_anchor_mode'])
        assert got == EXPECT[arm], f'{arm} s{s}: config {got}, expected {EXPECT[arm]}'
print(f'{len(ARMS) * len(SEEDS)} checkpoints OK at step {STEP}')

sha = set()
for arm, d in STATS.items():
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == EXPECT[arm][:2], f'{arm}: bake {d} is {(m["crn"], m["motion"])}'
    assert (d / 'v2_input_stats.npz').is_file(), f'{arm}: no stats in {d}'
    sha.add(m['train_ids_sha1'])
assert len(sha) == 1, f'the T2 bakes were fitted on different train splits: {sha}'
print(f'T2 stats: one train split {sha.pop()[:12]} for A1 / A2 / A3 / A3-ign')

assert CENSUS.is_file(), f'missing {CENSUS} (nexar_n0_census.ipynb)'
assert (DATA / constants.DEFS_FILENAME).is_file(), f'missing {DATA / constants.DEFS_FILENAME} (nexar_train step 3)'
VAL = load_split(constants.V2_SPLIT_NEXAR_VAL)
missing = [v for v in VAL if not (CLIP_S1 / f'{v}.npy').exists() or not (VIDEO_S1 / f'{v}.npy').exists()]
assert not missing, f'{len(missing)} nexar_val videos lack N2 features: {missing[:5]}'
print(f'nexar_val: {len(VAL)} videos, CLIP s1 + V2-S s1 present')

## 2. Stage `nexar_val` features to VM disk

In [ ]:
L_X = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_val', VAL)
L_U = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_val', VAL)
L_STATS = {arm: stage(d, LOCAL / 'stage' / f'stats_{arm}') for arm, d in STATS.items()}   # manifest + stats only
L_DATA = LOCAL / 'stage' / 'data'
L_DATA.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):
    shutil.copy2(f, L_DATA / f.name)

## 3. Score every arm (5 seeds, seed-averaged per frame)

One `nexar_eval` call per arm. A0 reads plain CLIP rows; A1 / A2 / A3 / A3-ign bake each item with **their own T2**
statistics (never Nexar's). An arm whose `nexar_readout.json` is on Drive is reused.

In [ ]:
for arm in ARMS:
    out = OUT / arm
    if (out / 'nexar_readout.json').exists():
        print(f'{arm}: reusing {out}')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', RUNS[arm] / f's{s}' / 'checkpoint_last.pt')]
    extra = []
    if arm in L_STATS:
        extra += ['--stats-dir', L_STATS[arm]]
        if EXPECT[arm][1] != OFF:
            extra += ['--video-s1-dir', L_U]
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.nexar_eval', *runs, '--census', CENSUS, '--clip-s1-dir', L_X, *extra,
         '--data-dir', L_DATA, '--split', constants.V2_SPLIT_NEXAR_VAL, '--out-dir', out],
        log=LOCAL / f'{arm}.log')
    print(f'{arm}: done in {time.time() - t0:.0f}s')

## 4. Regression gate — fresh A3-ign == the saved §28 zero-shot read-out

Same checkpoints, same stats, same tool: every number must match to 1e-6. A mismatch means the upload, the features or
the stats differ from what §28 read — stop and paste the two values to Claude.

In [ ]:
fresh = json.loads((OUT / 'A3ign' / 'nexar_readout.json').read_text())
if not (SAVED_ZS / 'nexar_readout.json').exists():
    print(f'no saved read-out at {SAVED_ZS} -- gate skipped (printed)')
else:
    saved = json.loads((SAVED_ZS / 'nexar_readout.json').read_text())
    pairs = [('whole macro', fresh['whole']['macro']['mean'], saved['whole']['macro']['mean']),
             ('whole micro', fresh['whole']['micro'], saved['whole']['micro']),
             ('clip AUC', fresh['whole']['clip_auc_max'], saved['whole']['clip_auc_max']),
             ('crop mean', fresh['crops']['mean_over_placements']['macro']['mean'],
              saved['crops']['mean_over_placements']['macro']['mean'])]
    for name, a, b in pairs:
        print(f'{name:12s} fresh {a:.6f}  saved {b:.6f}')
        assert abs(a - b) < 1e-6, f'A3-ign {name} differs from the §28 read: {a} vs {b}'
    print('gate PASS: A3-ign reproduces the saved zero-shot read')

## 5. Summary → paste `zeroshot_summary.md` to Claude

Per-arm numbers are each arm's own read-out. Contrasts are **paired by video** (the same crops / videos, 95 % bootstrap
over videos, 10,000 resamples): the crop endpoint per video = mean over its five placements (videos with all five), the
whole macro per positive video. Seeds are averaged inside each arm first, so these intervals do not include seed
variance. Printed only: nothing is decided.

In [ ]:
from core.metrics import cluster_bootstrap_ci
from core.tools.nexar_eval import CROP_SEP

R = {a: json.loads((OUT / a / 'nexar_readout.json').read_text()) for a in ARMS}
ITEM = {a: json.loads((OUT / a / 'item_aucs.json').read_text()) for a in ARMS}
P = [str(p) for p in constants.NEXAR_CROP_PLACEMENTS]


def fmt(c):
    return '—' if not c else f"{c['mean']:.4f} [{c['low']:.4f}, {c['high']:.4f}]"


def per_video(arm):
    crops, whole = {}, {}
    for name, auc in ITEM[arm].items():
        if CROP_SEP in name:
            v, _ = name.split(CROP_SEP)
            crops.setdefault(v, []).append(auc)
        else:
            whole[name] = auc
    return {v: float(np.mean(a)) for v, a in crops.items() if len(a) == len(P)}, whole


PV = {a: per_video(a) for a in ARMS}


def paired(a, b, k):
    x, y = PV[a][k], PV[b][k]
    d = {v: x[v] - y[v] for v in sorted(set(x) & set(y))}
    return cluster_bootstrap_ci(d, {v: v for v in d}, constants.V2_E1_BOOTSTRAP, constants.SEED, constants.V2_E1_CI)


lines = ['# Nexar zero-shot — T2-trained arms on nexar_val (225 videos, 112 positive; 5 seeds each, seed-averaged)', '',
         'Descriptive only (not pre-registered, decides nothing). Protocol N-B via `core.tools.nexar_eval`; '
         'each arm baked with its own T2 statistics. A0-A3 = E3 (`span`), A3-ign = E4-0 (`ignore`).', '',
         '| arm | crops mean (endpoint) | ' + ' | '.join(f'crop @{p}' for p in P)
         + ' | whole macro | whole micro | clip AUC |',
         '|---|---|' + '---|' * len(P) + '---|---|---|']
for a in ARMS:
    r = R[a]
    lines.append(f"| {a} | **{fmt(r['crops']['mean_over_placements']['macro'])}** | "
                 + ' | '.join(f"{r['crops'][p]['macro']['mean']:.4f}" for p in P)
                 + f" | {fmt(r['whole']['macro'])} | {r['whole']['micro']:.4f} | {r['whole']['clip_auc_max']:.4f} |")
w = R['A0']['whole']
lines += ['| middle ruler (position alone) | — | '
          + ' | '.join(f"{R['A0']['crops'][p]['middle_ruler_macro']['mean']:.4f}" for p in P)
          + f" | {w['middle_ruler_macro']['mean']:.4f} | clip oracle {w['clip_oracle_micro']:.4f} | — |", '',
          '## Per-seed whole macro', '']
for a in ARMS:
    lines.append(f"* {a}: { {k: round(v, 4) for k, v in sorted(R[a]['per_run_whole_macro'].items())} }")
lines += ['', '## Paired by video (printed)', '', '| contrast | crops mean Δ | whole macro Δ |', '|---|---|---|']
for a, b in [('A1', 'A0'), ('A2', 'A0'), ('A3', 'A0'), ('A3', 'A1'), ('A3', 'A2'), ('A3ign', 'A3'), ('A3ign', 'A0')]:
    lines.append(f'| {a} − {b} | {fmt(paired(a, b, 0))} | {fmt(paired(a, b, 1))} |')
lines += ['', f"Crops skipped (did not fit): {R['A0']['crops_skipped']}"]
text = '\n'.join(lines) + '\n'
(OUT / 'zeroshot_summary.md').write_text(text)
print(text)
print(f'written {OUT / "zeroshot_summary.md"}')